# First LLM Mesh Connection

## Learning Objectives

By completing this notebook, you will:
1. Connect to an LLM through Dataiku LLM Mesh
2. Make your first completion request
3. Understand request/response structure
4. Access usage metrics and costs
5. Handle basic errors

## Prerequisites

- Dataiku DSS 12.0+ with LLM Mesh enabled
- At least one LLM connection configured (see Admin)
- Basic Python knowledge

## Setup

This cell installs the Dataiku Python client and imports the `LLM` class that wraps the LLM Mesh API. After running it, you will have a configured `dataiku` session pointing at your DSS instance and a ready-to-use `LLM` handle for making governed completion requests. No API keys are needed here — authentication is managed centrally by LLM Mesh.

In [ ]:
# Import required libraries
import dataiku
from dataiku.llm import LLM
import json
from pprint import pprint

## Exercise 1: List Available Connections

First, let's see what LLM connections are available in this Dataiku instance.

In [ ]:
# Get API client
client = dataiku.api_client()

# List all connections
connections = client.list_connections()

# Filter to LLM connections only
llm_connections = [
    conn for conn in connections 
    if conn.get('type', '').lower() in ['anthropic', 'openai', 'azure_openai', 'llm']
]

print(f"Found {len(llm_connections)} LLM connection(s):\n")
for conn in llm_connections:
    print(f"  - {conn.get('name')}: {conn.get('type')}")

### 🎯 Exercise 1.1: Find Your Connection

If no connections are shown above, you need to configure one. Ask your instructor or administrator.

**Task:** Set the `CONNECTION_NAME` variable below to one of the available connections.

In [ ]:
# TODO: Replace with your connection name
CONNECTION_NAME = "claude-production"  # Change this!

# Test: Verify connection exists
assert CONNECTION_NAME in [c.get('name') for c in llm_connections], \
    f"Connection '{CONNECTION_NAME}' not found. Available: {[c.get('name') for c in llm_connections]}"

print(f"✓ Using connection: {CONNECTION_NAME}")

## Exercise 2: Your First LLM Request

Let's make a simple completion request.

In [ ]:
# Create LLM handle
llm = LLM(CONNECTION_NAME)

# Simple completion
response = llm.complete(
    prompt="What are the three main factors affecting crude oil prices?",
    max_tokens=200
)

print("Response:")
print(response.text)
print("\nMetadata:")
print(f"  Tokens used: {response.usage.total_tokens}")
print(f"  - Input: {response.usage.prompt_tokens}")
print(f"  - Output: {response.usage.completion_tokens}")

### 🎯 Exercise 2.1: Completion with Parameters

**Task:** Complete the function below to generate a summary with specific parameters:
- Max tokens: 100
- Temperature: 0.3 (more focused)
- The prompt should ask for a one-sentence summary

In [ ]:
def generate_summary(text: str) -> str:
    """
    Generate a one-sentence summary of the provided text.
    
    Args:
        text: Text to summarize
        
    Returns:
        One-sentence summary
    """
    # TODO: Create prompt asking for one-sentence summary
    prompt = f""  # Your code here
    
    # TODO: Call llm.complete() with appropriate parameters
    response = None  # Your code here
    
    return response.text.strip()

# Test data
test_text = """
The U.S. Energy Information Administration (EIA) reported that commercial crude oil 
inventories decreased by 5.2 million barrels during the week ending January 26. 
At 430.0 million barrels, U.S. crude oil inventories are about 3% below the five-year 
average for this time of year. The draw was larger than analyst expectations of 3.0 
million barrels, driven by strong refinery demand and lower imports.
"""

# Test your function
summary = generate_summary(test_text)
print("Summary:")
print(summary)

# Auto-graded test
assert summary is not None and len(summary) > 0, "Summary is empty"
assert len(summary.split('.')) <= 2, "Summary should be one sentence (or very short)"
assert "5.2 million" in summary or "decreased" in summary or "draw" in summary.lower(), \
    "Summary should mention the key fact (inventory draw)"

print("\n✓ Exercise 2.1 passed!")

## Exercise 3: Structured Output

LLMs can return structured data (JSON) for downstream processing.

In [ ]:
# Example: Extract structured data
report_text = """
WTI crude oil futures rose $2.45 to settle at $78.32 per barrel on Wednesday, 
following a larger-than-expected inventory draw reported by the EIA.
"""

prompt = f"""Extract the following information from this commodity report:
- Commodity name
- Price change (number and direction)
- Price level
- Key driver

Report: {report_text}

Return as JSON only, with keys: commodity, price_change, price_level, driver"""

response = llm.complete(prompt, max_tokens=150, temperature=0)

# Parse JSON
try:
    data = json.loads(response.text)
    print("Extracted data:")
    pprint(data)
except json.JSONDecodeError as e:
    print(f"Failed to parse JSON: {e}")
    print(f"Raw response: {response.text}")

### 🎯 Exercise 3.1: Extract Trading Signals

**Task:** Write a function that extracts a trading signal from a report. It should return:
- `direction`: "bullish", "bearish", or "neutral"
- `confidence`: "high", "medium", or "low"
- `reason`: one sentence explaining why

In [ ]:
def extract_trading_signal(report: str) -> dict:
    """
    Extract trading signal from commodity report.
    
    Args:
        report: Market report text
        
    Returns:
        Dict with keys: direction, confidence, reason
    """
    # TODO: Write prompt to extract signal
    prompt = f""""""  # Your code here
    
    # TODO: Get LLM response
    response = None  # Your code here
    
    # TODO: Parse JSON and return
    return {}  # Your code here

# Test cases
test_reports = [
    """Crude oil inventories plunged 7.2 million barrels, the largest draw in 
    six months, as refineries ramped up activity ahead of summer driving season.""",
    
    """OPEC+ announced a surprise production cut of 500,000 barrels per day, 
    tightening an already constrained market.""",
    
    """Oil prices moved sideways on mixed signals: inventory builds offset by 
    geopolitical tensions in the Middle East."""
]

for i, report in enumerate(test_reports, 1):
    signal = extract_trading_signal(report)
    print(f"\nReport {i}:")
    print(f"  Direction: {signal.get('direction')}")
    print(f"  Confidence: {signal.get('confidence')}")
    print(f"  Reason: {signal.get('reason')}")
    
    # Auto-graded tests
    assert 'direction' in signal, "Missing 'direction' field"
    assert signal['direction'] in ['bullish', 'bearish', 'neutral'], \
        f"Invalid direction: {signal['direction']}"
    assert 'confidence' in signal, "Missing 'confidence' field"
    assert 'reason' in signal, "Missing 'reason' field"

print("\n✓ Exercise 3.1 passed!")

## Exercise 4: Error Handling

Production code must handle errors gracefully.

In [ ]:
from dataiku.llm import RateLimitError, LLMError
import time

def robust_complete(prompt: str, max_retries: int = 3, **kwargs) -> str:
    """
    Make LLM request with error handling and retries.
    """
    llm = LLM(CONNECTION_NAME)
    
    for attempt in range(max_retries):
        try:
            response = llm.complete(prompt, **kwargs)
            return response.text
            
        except RateLimitError as e:
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt  # Exponential backoff
                print(f"Rate limited, waiting {wait_time}s...")
                time.sleep(wait_time)
            else:
                raise
                
        except LLMError as e:
            print(f"LLM error on attempt {attempt + 1}: {e}")
            if attempt >= max_retries - 1:
                raise
            time.sleep(1)
            
        except Exception as e:
            print(f"Unexpected error: {e}")
            raise
    
    raise Exception(f"Failed after {max_retries} attempts")

# Test
result = robust_complete(
    "What is the current OPEC production target?",
    max_tokens=100
)
print(result)

### 🎯 Exercise 4.1: Batch Processing with Error Handling

**Task:** Process a batch of prompts, skipping any that error but continuing with the rest.

In [ ]:
def batch_process(prompts: list[str]) -> list[dict]:
    """
    Process multiple prompts, handling errors gracefully.
    
    Args:
        prompts: List of prompt strings
        
    Returns:
        List of dicts with keys: prompt, response, success, error
    """
    results = []
    
    # TODO: Loop through prompts
    # TODO: Try to get completion for each
    # TODO: On success, append {prompt, response, success=True, error=None}
    # TODO: On error, append {prompt, response=None, success=False, error=str(e)}
    
    return results  # Your code here

# Test
test_prompts = [
    "What affects oil prices?",
    "Explain contango vs backwardation.",
    "What is OPEC+?"
]

results = batch_process(test_prompts)

# Display results
print(f"Processed {len(results)} prompts:\n")
for i, result in enumerate(results, 1):
    print(f"{i}. Success: {result['success']}")
    if result['success']:
        print(f"   Response: {result['response'][:100]}...")
    else:
        print(f"   Error: {result['error']}")

# Auto-graded tests
assert len(results) == len(test_prompts), "Should return one result per prompt"
assert all('success' in r for r in results), "All results should have 'success' field"
assert all('response' in r or 'error' in r for r in results), \
    "Results should have response or error"

success_count = sum(1 for r in results if r['success'])
print(f"\n✓ Exercise 4.1 passed! ({success_count}/{len(results)} succeeded)")

## Exercise 5: Cost Tracking

Monitor costs to stay within budget.

In [ ]:
class CostTracker:
    """Track LLM usage costs."""
    
    # Costs per 1M tokens (example values)
    COSTS = {
        'claude-sonnet-4': {'input': 3.0, 'output': 15.0},
        'gpt-4o': {'input': 2.5, 'output': 10.0},
        'gpt-3.5-turbo': {'input': 0.5, 'output': 1.5}
    }
    
    def __init__(self, model: str = 'claude-sonnet-4'):
        self.model = model
        self.total_cost = 0.0
        self.total_tokens = 0
        
    def record_usage(self, prompt_tokens: int, completion_tokens: int):
        """Record token usage and calculate cost."""
        costs = self.COSTS.get(self.model, {'input': 5.0, 'output': 15.0})
        
        cost = (
            prompt_tokens * costs['input'] / 1_000_000 +
            completion_tokens * costs['output'] / 1_000_000
        )
        
        self.total_cost += cost
        self.total_tokens += (prompt_tokens + completion_tokens)
        
        return cost
    
    def get_summary(self) -> dict:
        return {
            'total_cost_usd': self.total_cost,
            'total_tokens': self.total_tokens,
            'model': self.model
        }

# Usage example
tracker = CostTracker(model='claude-sonnet-4')

# Make some requests
for i in range(3):
    response = llm.complete(
        f"Question {i+1}: What are key oil price drivers?",
        max_tokens=100
    )
    
    cost = tracker.record_usage(
        response.usage.prompt_tokens,
        response.usage.completion_tokens
    )
    
    print(f"Request {i+1}: ${cost:.6f}")

# Summary
summary = tracker.get_summary()
print(f"\nTotal cost: ${summary['total_cost_usd']:.6f}")
print(f"Total tokens: {summary['total_tokens']:,}")

### 🎯 Exercise 5.1: Budget-Aware Processing

**Task:** Modify the batch processing function to stop when a budget limit is reached.

In [ ]:
def budget_aware_batch(prompts: list[str], budget_usd: float) -> dict:
    """
    Process prompts until budget is exhausted.
    
    Args:
        prompts: List of prompts
        budget_usd: Maximum cost in USD
        
    Returns:
        Dict with results and budget info
    """
    tracker = CostTracker(model='claude-sonnet-4')
    results = []
    
    # TODO: Process prompts
    # TODO: Track costs
    # TODO: Stop when budget would be exceeded
    
    return {
        'results': results,
        'processed': len(results),
        'total_prompts': len(prompts),
        'cost': tracker.total_cost,
        'budget': budget_usd,
        'within_budget': tracker.total_cost <= budget_usd
    }

# Test with small budget
test_prompts = [f"Question {i}" for i in range(100)]
output = budget_aware_batch(test_prompts, budget_usd=0.01)

print(f"Processed: {output['processed']}/{output['total_prompts']}")
print(f"Cost: ${output['cost']:.6f} (budget: ${output['budget']:.6f})")
print(f"Within budget: {output['within_budget']}")

# Auto-graded test
assert output['cost'] <= output['budget'] * 1.1, "Exceeded budget by >10%"
assert output['processed'] < output['total_prompts'], "Should stop before processing all"

print("\n✓ Exercise 5.1 passed!")

## Summary

Congratulations! You've completed your first LLM Mesh notebook. You've learned:

1. ✓ How to connect to LLMs through Dataiku LLM Mesh
2. ✓ Making completion requests with parameters
3. ✓ Extracting structured output (JSON)
4. ✓ Error handling and retry logic
5. ✓ Cost tracking and budget management

## Next Steps

- **Notebook 02**: Provider comparison and selection
- **Module 1**: Prompt design and optimization
- **Module 2**: RAG applications with Knowledge Banks

## Solutions

<details>
<summary>Exercise 2.1 Solution</summary>

```python
def generate_summary(text: str) -> str:
    prompt = f"""Provide a one-sentence summary of this text:
    
{text}

Summary:"""
    
    response = llm.complete(
        prompt=prompt,
        max_tokens=100,
        temperature=0.3
    )
    
    return response.text.strip()
```
</details>

<details>
<summary>Exercise 3.1 Solution</summary>

```python
def extract_trading_signal(report: str) -> dict:
    prompt = f"""Analyze this commodity report and extract a trading signal.

Report: {report}

Return JSON with:
- direction: "bullish", "bearish", or "neutral"
- confidence: "high", "medium", or "low"
- reason: one sentence explanation

JSON only:"""
    
    response = llm.complete(prompt, max_tokens=150, temperature=0)
    return json.loads(response.text)
```
</details>

<details>
<summary>Exercise 4.1 Solution</summary>

```python
def batch_process(prompts: list[str]) -> list[dict]:
    results = []
    llm = LLM(CONNECTION_NAME)
    
    for prompt in prompts:
        try:
            response = llm.complete(prompt, max_tokens=100)
            results.append({
                'prompt': prompt,
                'response': response.text,
                'success': True,
                'error': None
            })
        except Exception as e:
            results.append({
                'prompt': prompt,
                'response': None,
                'success': False,
                'error': str(e)
            })
    
    return results
```
</details>

<details>
<summary>Exercise 5.1 Solution</summary>

```python
def budget_aware_batch(prompts: list[str], budget_usd: float) -> dict:
    tracker = CostTracker(model='claude-sonnet-4')
    results = []
    llm = LLM(CONNECTION_NAME)
    
    for prompt in prompts:
        # Estimate cost (rough)
        estimated_cost = (len(prompt) / 4) * 3.0 / 1_000_000 + \
                        (100) * 15.0 / 1_000_000
        
        if tracker.total_cost + estimated_cost > budget_usd:
            break
        
        try:
            response = llm.complete(prompt, max_tokens=100)
            cost = tracker.record_usage(
                response.usage.prompt_tokens,
                response.usage.completion_tokens
            )
            results.append(response.text)
        except Exception as e:
            continue
    
    return {
        'results': results,
        'processed': len(results),
        'total_prompts': len(prompts),
        'cost': tracker.total_cost,
        'budget': budget_usd,
        'within_budget': tracker.total_cost <= budget_usd
    }
```
</details>